In [ ]:
# 1/4 - clone main + prototype (fresh, fault-tolerant)
import subprocess, os
print('===== NATIVE FLASH ALL 7 KERNELS TEST (PURE TKV) =====')
for d,br in [('/content/TVmain','main'),('/content/TVcpu','prototype/cuda-backend')]:
    subprocess.run('rm -rf '+d, shell=True)
    subprocess.run('git clone --depth 1 --branch '+br+' https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
    print(br, subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())

In [ ]:
# 2/4 - build native .so + flash_all .so. NO C injection. CUNG flags.
import subprocess, os
FLAGS='-O2 -mavx2 -mfma'
root='/content/TVmain/Release/tools/simd_native'
srcs='tkv_simd_native.c tkv_cuda_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c tkv_prof.c tkv_attn_cuda.c'
p=subprocess.run('cd '+root+' && gcc '+FLAGS+' -shared -fPIC '+srcs+' -lm -lpthread -ldl -o tkv_simd_native.so', shell=True, capture_output=True, text=True, timeout=600)
print('native build rc=',p.returncode)
if p.returncode!=0: print(p.stderr[:2000])
ex='/content/TVcpu/Release/cuda_backend/examples'
cc=ex+'/flash_all_cpu.cpu.c'; so=ex+'/flash_all_cpu.so'
txt=open(cc).read()
print('C INJECTED =', 'tkv_flash' in txt, '(expect False)')
print('exports =', txt.count('EXPORT int'))
p2=subprocess.run('gcc '+FLAGS+' -shared -fPIC '+cc+' -o '+so+' -lm -lpthread -ldl', shell=True, capture_output=True, text=True, timeout=600)
print('tkv C build rc=',p2.returncode)
if p2.returncode!=0: print(p2.stderr[:3000])

In [ ]:
# 3/4 - compare all 7 flash kernels (maxabs + bitwise info)
import ctypes, numpy as np
P=ctypes.POINTER(ctypes.c_float); I=ctypes.c_int; D=ctypes.c_double
libn=ctypes.CDLL('/content/TVmain/Release/tools/simd_native/tkv_simd_native.so')
libt=ctypes.CDLL('/content/TVcpu/Release/cuda_backend/examples/flash_all_cpu.so')
n=64; h=256; nh=h//64; hd=64; head=2; inv=0.125
rng=np.random.default_rng(61)
qkv=rng.standard_normal(n*3*h).astype(np.float32)
dctx=rng.standard_normal(n*h).astype(np.float32)
def ptr(a): return a.ctypes.data_as(P)
def cmp(name,fn_a,fn_b,arrays):
    for a in arrays: pass
    fn_a(); fn_b()
    for nm,x,y in arrays:
        d=np.count_nonzero(x.view(np.uint32)!=y.view(np.uint32))
        print('%s %s bitwise=%s diff=%d/%d maxabs=%.3g'%(name,nm,d==0,d,x.size,np.abs(x-y).max()))
# 1 FwdLse
ctx_a=np.zeros(n*64,np.float32); ctx_b=ctx_a.copy()
lse_a=np.zeros(n,np.float32); lse_b=lse_a.copy()
args=[P,I,P,I,P,I,I,I,I,I,D]
na=libn.FlashAttentionFwdLse; na.argtypes=args; na.restype=I
tb=libt.FlashAttentionFwdLse; tb.argtypes=args; tb.restype=I
na(ptr(qkv),qkv.size,ptr(ctx_a),ctx_a.size,ptr(lse_a),lse_a.size,n,h,hd,head,inv)
tb(ptr(qkv),qkv.size,ptr(ctx_b),ctx_b.size,ptr(lse_b),lse_b.size,n,h,hd,head,inv)
cmp('FwdLse',lambda:None,lambda:None,[('ctx',ctx_a,ctx_b),('lse',lse_a,lse_b)])
# 2 FwdFused
fa=np.zeros(n*64,np.float32); fb=fa.copy()
args=[P,I,P,I,I,I,I,I,D]
na=libn.FlashAttentionFwdFused; na.argtypes=args; na.restype=I
tb=libt.FlashAttentionFwdFused; tb.argtypes=args; tb.restype=I
na(ptr(qkv),qkv.size,ptr(fa),fa.size,n,h,hd,head,inv)
tb(ptr(qkv),qkv.size,ptr(fb),fb.size,n,h,hd,head,inv)
cmp('FwdFused',lambda:None,lambda:None,[('ctx',fa,fb)])
# 3 BwdFusedLse
c_n=np.zeros(n*64,np.float32); l_n=np.zeros(n,np.float32)
na=libn.FlashAttentionFwdLse; na.argtypes=[P,I,P,I,P,I,I,I,I,I,D]
na(ptr(qkv),qkv.size,ptr(c_n),c_n.size,ptr(l_n),l_n.size,n,h,hd,head,inv)
oa=np.zeros(n*64,np.float32); oc=oa.copy(); qa=np.zeros(n*192,np.float32); qc=qa.copy()
args=[P,I,P,I,P,I,P,I,P,I,P,I,I,I,I,I,D]
na=libn.FlashAttentionBwdFusedLse; na.argtypes=args; na.restype=I
tb=libt.FlashAttentionBwdFusedLse; tb.argtypes=args; tb.restype=I
na(ptr(qkv),qkv.size,ptr(dctx),dctx.size,ptr(c_n),c_n.size,ptr(l_n),l_n.size,ptr(oa),oa.size,ptr(qa),qa.size,n,h,hd,head,inv)
tb(ptr(qkv),qkv.size,ptr(dctx),dctx.size,ptr(c_n),c_n.size,ptr(l_n),l_n.size,ptr(oc),oc.size,ptr(qc),qc.size,n,h,hd,head,inv)
cmp('BwdLse',lambda:None,lambda:None,[('ctxo',oa,oc),('dqo',qa,qc)])
# 4 BwdFused
ba=np.zeros(n*64,np.float32); bb=ba.copy(); ga=np.zeros(n*192,np.float32); gb=ga.copy()
args=[P,I,P,I,P,I,P,I,I,I,I,I,I,D]
na=libn.FlashAttentionBwdFused; na.argtypes=args; na.restype=I
tb=libt.FlashAttentionBwdFused; tb.argtypes=args; tb.restype=I
na(ptr(qkv),qkv.size,ptr(dctx),dctx.size,ptr(ba),ba.size,ptr(ga),ga.size,n,h,hd,head,inv)
tb(ptr(qkv),qkv.size,ptr(dctx),dctx.size,ptr(bb),bb.size,ptr(gb),gb.size,n,h,hd,head,inv)
cmp('BwdFused',lambda:None,lambda:None,[('ctxo',ba,bb),('dqo',ga,gb)])
# 5 FwdLseMulti
ma=np.zeros(nh*n*64,np.float32); mb=ma.copy(); la=np.zeros(nh*n,np.float32); lb=la.copy()
args=[P,I,P,I,P,I,I,I,I,D,I]
na=libn.FlashAttentionFwdLseMulti; na.argtypes=args; na.restype=I
tb=libt.FlashAttentionFwdLseMulti; tb.argtypes=args; tb.restype=I
na(ptr(qkv),qkv.size,ptr(ma),ma.size,ptr(la),la.size,n,h,hd,inv,1)
tb(ptr(qkv),qkv.size,ptr(mb),mb.size,ptr(lb),lb.size,n,h,hd,inv,1)
cmp('FwdLseMulti',lambda:None,lambda:None,[('ctx',ma,mb),('lse',la,lb)])
# 6 BwdFusedLseMulti
cM=np.zeros(nh*n*64,np.float32); lM=np.zeros(nh*n,np.float32)
na=libn.FlashAttentionFwdLseMulti; na.argtypes=args; na.restype=I
na(ptr(qkv),qkv.size,ptr(cM),cM.size,ptr(lM),lM.size,n,h,hd,inv,1)
xa=np.zeros(nh*n*64,np.float32); xb=xa.copy(); ya=np.zeros(nh*n*192,np.float32); yb=ya.copy()
args=[P,I,P,I,P,I,P,I,P,I,P,I,I,I,I,D,I]
na=libn.FlashAttentionBwdFusedLseMulti; na.argtypes=args; na.restype=I
tb=libt.FlashAttentionBwdFusedLseMulti; tb.argtypes=args; tb.restype=I
na(ptr(qkv),qkv.size,ptr(dctx),dctx.size,ptr(cM),cM.size,ptr(lM),lM.size,ptr(xa),xa.size,ptr(ya),ya.size,n,h,hd,inv,1)
tb(ptr(qkv),qkv.size,ptr(dctx),dctx.size,ptr(cM),cM.size,ptr(lM),lM.size,ptr(xb),xb.size,ptr(yb),yb.size,n,h,hd,inv,1)
cmp('BwdLseMulti',lambda:None,lambda:None,[('ctxo',xa,xb),('dqo',ya,yb)])
# 7 BwdMergeMulti
ta=np.zeros(n*h,np.float32); tb2=ta.copy(); da=np.zeros(n*3*h,np.float32); db=da.copy()
args=[P,I,P,I,P,I,P,I,P,I,P,I,I,I,I,D,I]
na=libn.FlashAttentionBwdMergeMulti; na.argtypes=args; na.restype=I
tb=libt.FlashAttentionBwdMergeMulti; tb.argtypes=args; tb.restype=I
na(ptr(qkv),qkv.size,ptr(dctx),dctx.size,ptr(cM),cM.size,ptr(lM),lM.size,ptr(ta),ta.size,ptr(da),da.size,n,h,hd,inv,1)
tb(ptr(qkv),qkv.size,ptr(dctx),dctx.size,ptr(cM),cM.size,ptr(lM),lM.size,ptr(tb2),tb2.size,ptr(db),db.size,n,h,hd,inv,1)
cmp('BwdMergeMulti',lambda:None,lambda:None,[('ctxT',ta,tb2),('dqkv',da,db)])

In [ ]:
# 4/4 - summary
print('Flash path waiver s0.125. Compare maxabs per kernel.')